# Support Vector Classifier (SVC) — Classification Practical

This notebook follows the SVC items in the provided ML practical cheat sheet and adds evaluation, tuning, visualizations, and significance testing.

**Cheat-sheet hyperparameters included**
- `kernel`: `["linear", "poly", "rbf", "sigmoid"]`
- `C`: `[0.1, 1, 10]`
- `gamma`: `["scale", "auto"]` for `poly`, `rbf`, and `sigmoid`
- `degree`: `[2, 3]` for `poly`
- `probability`: `[True, False]`
- `random_state`: fixed at `42` (meaningful to SVC's probability estimation when enabled)

The grid is expressed as conditional parameter dictionaries so `gamma` is not needlessly searched for the linear kernel and `degree` is only searched for the polynomial kernel. Grid search uses macro-F1 as the primary selection metric. SVC is sensitive to feature scale, so scaling is included in the preprocessing pipeline.

**How to use:** Run cells from top to bottom. When prompted, enter the CSV file path and target column. This notebook is syntax-checked but has not been executed against your dataset; runtime depends on dataset size and grid-search folds.


In [ ]:
# =========================
# 1. Imports and configuration
# =========================
import os, glob, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ttest_rel, wilcoxon
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, cohen_kappa_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, log_loss, roc_auc_score,
    average_precision_score, RocCurveDisplay, PrecisionRecallDisplay
)
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, GridSearchCV, cross_val_score,
    learning_curve
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, label_binarize
from sklearn.svm import SVC
from sklearn.utils.multiclass import type_of_target

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Libraries loaded. Random state =", RANDOM_STATE)

# =========================
# 2. Load dataset
# =========================
csv_files = glob.glob("*.csv")
if len(csv_files) == 1:
    default_path = csv_files[0]
    print("Detected CSV:", default_path)
    path = input("Press Enter to use this file, or type another CSV path: ").strip() or default_path
elif len(csv_files) > 1:
    print("CSV files in current folder:", csv_files)
    path = input("Enter the CSV path to use: ").strip()
else:
    path = input("Enter the path to your CSV dataset: ").strip()

if not os.path.exists(path):
    raise FileNotFoundError(f"CSV file not found: {path}")
df = pd.read_csv(path)
print("Dataset shape:", df.shape)
display(df.head())
print("\nData types and non-null counts:")
df.info()
print("\nNumeric summary:")
display(df.describe(include="all").T)
print("\nMissing values (top 30):")
display(df.isna().sum().sort_values(ascending=False).head(30))
print("\nDuplicate rows:", df.duplicated().sum())

# =========================
# 3. Choose target and basic cleaning
# =========================
target = input("Enter the exact target/class column name: ").strip()
if target not in df.columns:
    raise ValueError(f"Target '{target}' not found. Available columns: {list(df.columns)}")

df = df.drop_duplicates().dropna(subset=[target]).copy()
y = df[target].copy()
X = df.drop(columns=[target]).copy()

# Remove likely identifier columns (heuristic; review this list before relying on results).
id_like = [c for c in X.columns if (
    c.lower() in {"id", "index", "row_id", "customerid", "customer_id", "record_id"} or
    (X[c].nunique(dropna=True) >= 0.98 * max(len(X), 1) and
     (pd.api.types.is_integer_dtype(X[c]) or X[c].dtype == "object"))
)]
if id_like:
    print("Potential identifier columns excluded (review if meaningful):", id_like)
    X = X.drop(columns=id_like)

if y.nunique() < 2:
    raise ValueError("Classification requires at least two target classes.")
if y.nunique() > 50 and y.nunique() > 0.2 * len(y):
    print("WARNING: target has many unique values. Confirm this is a classification target.")

print("Feature matrix:", X.shape, "| Classes:", y.nunique())
display(y.value_counts(dropna=False).rename("count").to_frame())

# Target distribution
plt.figure(figsize=(9, 4))
y.astype(str).value_counts().plot(kind="bar")
plt.title("Target class distribution")
plt.xlabel("Class"); plt.ylabel("Count"); plt.xticks(rotation=45, ha="right")
plt.tight_layout(); plt.show()

# Numeric distributions and correlation
num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]
if num_cols:
    X[num_cols].hist(figsize=(12, max(4, 2.5 * ((len(num_cols)+3)//4))), bins=20)
    plt.suptitle("Numeric feature distributions"); plt.tight_layout(); plt.show()
    if len(num_cols) >= 2:
        plt.figure(figsize=(10, 7))
        corr = X[num_cols].corr(numeric_only=True)
        plt.imshow(corr, aspect="auto")
        plt.colorbar(label="Correlation")
        plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
        plt.yticks(range(len(corr.index)), corr.index)
        plt.title("Numeric feature correlation matrix")
        plt.tight_layout(); plt.show()
if cat_cols:
    for col in cat_cols[:6]:
        plt.figure(figsize=(8, 3))
        X[col].astype(str).value_counts().head(15).plot(kind="bar")
        plt.title(f"Top categories: {col}")
        plt.ylabel("Count"); plt.xticks(rotation=45, ha="right")
        plt.tight_layout(); plt.show()

# =========================
# 4. Safe preprocessing pipelines
# =========================
# Keep preprocessing inside Pipeline to prevent data leakage.
try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:  # compatibility with older scikit-learn
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", ohe)
])
transformers = []
if num_cols:
    transformers.append(("num", numeric_transformer, num_cols))
if cat_cols:
    transformers.append(("cat", categorical_transformer, cat_cols))
preprocessor = ColumnTransformer(transformers=transformers, remainder="drop")

# PCA comparison: PCA only on numeric features, categorical one-hot features pass through.
# If fewer than 2 numeric features exist, the PCA branch is skipped.
if len(num_cols) >= 2:
    pca_preprocessor = ColumnTransformer(transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("pca", PCA(n_components=0.95, random_state=RANDOM_STATE))
        ]), num_cols),
        ("cat", categorical_transformer, cat_cols)
    ], remainder="drop")
    pca_available = True
else:
    pca_preprocessor = None
    pca_available = False
    print("PCA comparison will be skipped: fewer than two numeric features.")

# =========================
# 5. Train/test split and CV setup
# =========================
class_counts = y.value_counts()
min_class_count = int(class_counts.min())
if min_class_count < 2:
    raise ValueError("At least one class has fewer than 2 rows; stratified split/CV is not reliable.")
test_fraction = max(0.2, min(0.35, 1.0 - (2 * y.nunique() / len(y))))
test_fraction = min(test_fraction, 0.5)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_fraction, random_state=RANDOM_STATE, stratify=y
)
min_train_class = int(y_train.value_counts().min())
outer_splits = min(5, min_train_class)
if outer_splits < 2:
    raise ValueError("Not enough training examples per class for cross-validation.")
inner_splits = min(3, max(2, min_train_class))
outer_cv = StratifiedKFold(n_splits=outer_splits, shuffle=True, random_state=RANDOM_STATE)
inner_cv = StratifiedKFold(n_splits=inner_splits, shuffle=True, random_state=RANDOM_STATE)
print(f"Train: {X_train.shape}; test: {X_test.shape}; outer CV folds: {outer_splits}; inner CV folds: {inner_splits}")

# =========================
# 6. SVC and full cheat-sheet parameter grid
# =========================
# Conditional grids prevent irrelevant parameters from being searched for other kernels.
# `probability` is included as listed in the cheat sheet; probability=True enables predict_proba
# but adds fitting cost. random_state is fixed at 42 for reproducibility.
svc = SVC(random_state=RANDOM_STATE, decision_function_shape="ovr")
pipeline = Pipeline([("prep", preprocessor), ("model", svc)])

param_grid = [
    {
        "model__kernel": ["linear"],
        "model__C": [0.1, 1, 10],
        "model__probability": [True, False],
    },
    {
        "model__kernel": ["poly"],
        "model__C": [0.1, 1, 10],
        "model__gamma": ["scale", "auto"],
        "model__degree": [2, 3],
        "model__probability": [True, False],
    },
    {
        "model__kernel": ["rbf", "sigmoid"],
        "model__C": [0.1, 1, 10],
        "model__gamma": ["scale", "auto"],
        "model__probability": [True, False],
    },
]
print("Conditional parameter grid prepared.")
print("Grid includes all cheat-sheet values: kernels, C, gamma, polynomial degree, probability.")
print("SVC random_state is fixed to", RANDOM_STATE)

# =========================
# 7. Nested cross-validation for unbiased model comparison
# =========================
def make_pipeline(use_pca=False):
    prep = pca_preprocessor if (use_pca and pca_available) else preprocessor
    return Pipeline([("prep", prep), ("model", SVC(
        random_state=RANDOM_STATE, decision_function_shape="ovr"
    ))])

def safe_metric_bundle(y_true, y_pred, y_score=None, labels=None):
    result = {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "precision_weighted": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_weighted": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "matthews_corrcoef": matthews_corrcoef(y_true, y_pred),
        "cohen_kappa": cohen_kappa_score(y_true, y_pred),
    }
    if y_score is not None:
        try:
            if len(labels) == 2:
                positive_scores = y_score[:, 1] if getattr(y_score, "ndim", 1) == 2 else y_score
                result["roc_auc"] = roc_auc_score(y_true, positive_scores)
                result["average_precision"] = average_precision_score(y_true, positive_scores)
            else:
                result["roc_auc_ovr_macro"] = roc_auc_score(
                    y_true, y_score, labels=labels, multi_class="ovr", average="macro"
                )
                result["average_precision_macro"] = average_precision_score(
                    label_binarize(y_true, classes=labels), y_score, average="macro"
                )
        except Exception:
            pass
    return result

def get_score_for_svc(estimator, X_part):
    # Prefer calibrated probabilities where available; otherwise use decision scores.
    model = estimator.named_steps["model"]
    if hasattr(model, "predict_proba") and model.probability:
        return estimator.predict_proba(X_part)
    return estimator.decision_function(X_part)

def nested_cv_evaluate(use_pca=False):
    fold_rows = []
    for fold, (tr_idx, val_idx) in enumerate(outer_cv.split(X_train, y_train), start=1):
        X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
        search = GridSearchCV(
            make_pipeline(use_pca),
            param_grid=param_grid,
            scoring="f1_macro",
            cv=inner_cv,
            n_jobs=-1,
            refit=True,
            error_score=np.nan,
            return_train_score=True
        )
        start = time.time()
        search.fit(X_tr, y_tr)
        best = search.best_estimator_
        pred = best.predict(X_val)
        try:
            score = get_score_for_svc(best, X_val)
        except Exception:
            score = None
        row = safe_metric_bundle(y_val, pred, score, labels=list(best.named_steps["model"].classes_))
        row["fold"] = fold
        row["best_params"] = search.best_params_
        row["best_inner_f1_macro"] = search.best_score_
        row["fit_seconds"] = time.time() - start
        fold_rows.append(row)
        print(f"Fold {fold}/{outer_splits} | macro-F1={row['f1_macro']:.4f} | {row['fit_seconds']:.1f}s")
    return pd.DataFrame(fold_rows)

print("Running nested CV without PCA. This may take time because SVC probability=True is computationally expensive.")
nested_no_pca = nested_cv_evaluate(use_pca=False)
display(nested_no_pca)
print("\nNested-CV mean metrics (no PCA):")
display(nested_no_pca.select_dtypes(include=np.number).drop(columns=["fold"], errors="ignore").mean().to_frame("mean"))

if pca_available:
    print("Running matched-fold nested CV with PCA.")
    nested_pca = nested_cv_evaluate(use_pca=True)
    display(nested_pca)
    print("\nNested-CV mean metrics (PCA):")
    display(nested_pca.select_dtypes(include=np.number).drop(columns=["fold"], errors="ignore").mean().to_frame("mean"))
else:
    nested_pca = None

# =========================
# 8. Paired significance tests: PCA vs no-PCA macro-F1
# =========================
if nested_pca is not None:
    a = nested_no_pca["f1_macro"].to_numpy()
    b = nested_pca["f1_macro"].to_numpy()
    print("Paired fold-level macro-F1 values")
    print("No PCA:", np.round(a, 4))
    print("PCA   :", np.round(b, 4))
    print("Mean difference (PCA - no PCA):", round(float(np.mean(b-a)), 6))
    if len(a) >= 2:
        print("Paired t-test:", ttest_rel(b, a, nan_policy="omit"))
        try:
            print("Wilcoxon signed-rank:", wilcoxon(b, a, zero_method="wilcox"))
        except Exception as e:
            print("Wilcoxon test unavailable:", e)
    print("Caution: few CV folds give low statistical power; treat p-values as supporting evidence, not proof.")
else:
    print("PCA significance test skipped because PCA branch was unavailable.")

# =========================
# 9. Final GridSearchCV on the training split
# =========================
final_search = GridSearchCV(
    make_pipeline(use_pca=False),
    param_grid=param_grid,
    scoring={
        "accuracy": "accuracy",
        "precision_macro": "precision_macro",
        "recall_macro": "recall_macro",
        "f1_macro": "f1_macro",
        "f1_weighted": "f1_weighted",
        "balanced_accuracy": "balanced_accuracy",
    },
    refit="f1_macro",
    cv=inner_cv,
    n_jobs=-1,
    error_score=np.nan,
    return_train_score=True
)
start = time.time()
final_search.fit(X_train, y_train)
best_model = final_search.best_estimator_
print(f"Final grid search time: {time.time()-start:.1f}s")
print("Best parameters:", final_search.best_params_)
print("Best mean CV macro-F1:", round(final_search.best_score_, 5))
results = pd.DataFrame(final_search.cv_results_).sort_values("rank_test_f1_macro")
display(results[[
    "rank_test_f1_macro", "mean_test_f1_macro", "std_test_f1_macro",
    "mean_test_accuracy", "mean_test_precision_macro", "mean_test_recall_macro",
    "mean_test_f1_weighted", "mean_test_balanced_accuracy", "params"
]].head(20))

plt.figure(figsize=(9, 4))
plt.plot(results["rank_test_f1_macro"].head(30), results["mean_test_f1_macro"].head(30), marker="o")
plt.title("Top SVC grid-search candidates")
plt.xlabel("Rank (top 30 candidates)"); plt.ylabel("Mean CV macro-F1")
plt.tight_layout(); plt.show()

# =========================
# 10. Held-out test evaluation
# =========================
y_pred = best_model.predict(X_test)
try:
    y_score = get_score_for_svc(best_model, X_test)
except Exception:
    y_score = None
classes = list(best_model.named_steps["model"].classes_)
test_metrics = safe_metric_bundle(y_test, y_pred, y_score, labels=classes)
if y_score is not None and hasattr(best_model.named_steps["model"], "predict_proba") and best_model.named_steps["model"].probability:
    try:
        test_metrics["log_loss"] = log_loss(y_test, y_score, labels=classes)
    except Exception:
        pass
print("Held-out test metrics:")
display(pd.Series(test_metrics, name="test_score").to_frame())
print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))

# Confusion matrix
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax, xticks_rotation=45, cmap="Blues")
ax.set_title("SVC confusion matrix")
plt.tight_layout(); plt.show()

# ROC and precision-recall plots. Use probabilities when available, decision scores otherwise.
if y_score is not None:
    try:
        if len(classes) == 2:
            positive_score = y_score[:, 1] if np.ndim(y_score) == 2 else y_score
            RocCurveDisplay.from_predictions(y_test, positive_score)
            plt.title("SVC ROC curve"); plt.tight_layout(); plt.show()
            PrecisionRecallDisplay.from_predictions(y_test, positive_score)
            plt.title("SVC precision-recall curve"); plt.tight_layout(); plt.show()
        elif np.ndim(y_score) == 2:
            y_bin = label_binarize(y_test, classes=classes)
            plt.figure(figsize=(8, 6))
            for i, cls in enumerate(classes):
                if len(np.unique(y_bin[:, i])) == 2:
                    from sklearn.metrics import roc_curve
                    fpr, tpr, _ = roc_curve(y_bin[:, i], y_score[:, i])
                    plt.plot(fpr, tpr, label=f"Class {cls}")
            plt.plot([0, 1], [0, 1], linestyle="--")
            plt.xlabel("False positive rate"); plt.ylabel("True positive rate")
            plt.title("Multiclass one-vs-rest ROC curves"); plt.legend()
            plt.tight_layout(); plt.show()
    except Exception as e:
        print("ROC/PR plot could not be generated for this dataset/model:", e)
else:
    print("ROC/PR curves skipped: model did not provide usable decision/probability scores.")

# =========================
# 11. Learning curve
# =========================
try:
    lc_cv = StratifiedKFold(n_splits=min(3, max(2, int(y_train.value_counts().min()))),
                            shuffle=True, random_state=RANDOM_STATE)
    sizes, train_scores, valid_scores = learning_curve(
        clone(make_pipeline(False)).set_params(**{
            k: v for k, v in final_search.best_params_.items()
        }),
        X_train, y_train, cv=lc_cv, scoring="f1_macro",
        train_sizes=np.linspace(0.3, 1.0, 4), n_jobs=-1, error_score=np.nan
    )
    plt.figure(figsize=(8, 5))
    plt.plot(sizes, np.nanmean(train_scores, axis=1), marker="o", label="Training macro-F1")
    plt.plot(sizes, np.nanmean(valid_scores, axis=1), marker="o", label="Validation macro-F1")
    plt.xlabel("Training examples"); plt.ylabel("Macro-F1")
    plt.title("SVC learning curve"); plt.legend(); plt.tight_layout(); plt.show()
except Exception as e:
    print("Learning curve skipped:", e)

# =========================
# 12. Save test predictions and summary
# =========================
predictions = pd.DataFrame({"actual": y_test.reset_index(drop=True),
                            "predicted": pd.Series(y_pred).reset_index(drop=True)})
predictions.to_csv("svc_test_predictions.csv", index=False)
summary = pd.DataFrame([{
    "model": "SVC",
    "best_params": str(final_search.best_params_),
    "best_cv_macro_f1": final_search.best_score_,
    **test_metrics
}])
summary.to_csv("svc_classification_summary.csv", index=False)
print("Saved: svc_test_predictions.csv and svc_classification_summary.csv")
print("Notebook complete.")
